In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder.appName("NYC").getOrCreate()
base_path = "/home/jovyan/work"

In [11]:
# df = spark.read.parquet(f"{base_path}/permitted_event_hist_partitioned/year=2020/month=1/c5fea84569df4a7a96a0a41baf5b2106-0.parquet")
df = spark.read.parquet(f"{base_path}/permitted_event_hist_partitioned/")

df_filtered = df.filter((df.year == 2020) & (df.month == 1))

In [12]:
#TODO: change filter to not only one month but entire year or more

In [13]:
df.printSchema()

root
 |-- Event ID: double (nullable = true)
 |-- Event Name: string (nullable = true)
 |-- Start Date/Time: string (nullable = true)
 |-- End Date/Time: string (nullable = true)
 |-- Event Agency: string (nullable = true)
 |-- Event Type: string (nullable = true)
 |-- Event Borough: string (nullable = true)
 |-- Event Location: string (nullable = true)
 |-- Event Street Side: string (nullable = true)
 |-- Street Closure Type: string (nullable = true)
 |-- Community Board: string (nullable = true)
 |-- Police Precinct: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [14]:
df_filtered.show(3, vertical=True)

-RECORD 0-----------------------------------
 Event ID            | 514607.0             
 Event Name          | Celebration          
 Start Date/Time     | 01/20/2020 11:00:... 
 End Date/Time       | 01/20/2020 12:00:... 
 Event Agency        | Parks Department     
 Event Type          | Special Event        
 Event Borough       | Manhattan            
 Event Location      | Central Park: Wag... 
 Event Street Side   |                      
 Street Closure Type | N/A                  
 Community Board     | 64,                  
 Police Precinct     | 22,                  
 year                | 2020                 
 month               | 1                    
-RECORD 1-----------------------------------
 Event ID            | 514607.0             
 Event Name          | Celebration          
 Start Date/Time     | 01/20/2020 11:00:... 
 End Date/Time       | 01/20/2020 12:00:... 
 Event Agency        | Parks Department     
 Event Type          | Special Event        
 Event Bor

In [15]:
# --- remove potential special chars (ASCII) like e.g. Ã¢â¬â ---
# --- drop all duplicate cols ---

for col in ['Event Name', 'Event Agency', 'Event Type', 'Event Borough', 'Event Location']:
    df_filtered = df_filtered.withColumn(col, regexp_replace(col, '[^\x00-\x7F]', ''))

df_filtered = df_filtered.dropDuplicates()

df_filtered.show(3, vertical=True)

-RECORD 0-----------------------------------
 Event ID            | 516645.0             
 Event Name          | Lawn Maintenance-... 
 Start Date/Time     | 01/02/2020 12:01:... 
 End Date/Time       | 01/02/2020 11:59:... 
 Event Agency        | Parks Department     
 Event Type          | Special Event        
 Event Borough       | Manhattan            
 Event Location      | Madison Square Pa... 
 Event Street Side   |                      
 Street Closure Type | N/A                  
 Community Board     | 5,                   
 Police Precinct     | 13,                  
 year                | 2020                 
 month               | 1                    
-RECORD 1-----------------------------------
 Event ID            | 517686.0             
 Event Name          |  Football - Youth    
 Start Date/Time     | 01/03/2020 08:00:... 
 End Date/Time       | 01/03/2020 10:00:... 
 Event Agency        | Parks Department     
 Event Type          | Sport - Adult        
 Event Bor

In [17]:
df_filtered.sort("Start Date/Time", ascending=True).show(3, vertical=True)

-RECORD 0-----------------------------------
 Event ID            | 515210.0             
 Event Name          | Menorah              
 Start Date/Time     | 01/01/2020 01:00:... 
 End Date/Time       | 01/01/2020 04:00:... 
 Event Agency        | Parks Department     
 Event Type          | Special Event        
 Event Borough       | Brooklyn             
 Event Location      | Prospect Park: Wi... 
 Event Street Side   |                      
 Street Closure Type | N/A                  
 Community Board     | 55,                  
 Police Precinct     | 78,                  
 year                | 2020                 
 month               | 1                    
-RECORD 1-----------------------------------
 Event ID            | 519641.0             
 Event Name          | Virgen de la Nube... 
 Start Date/Time     | 01/01/2020 01:15:... 
 End Date/Time       | 01/01/2020 02:30:... 
 Event Agency        | Police Department    
 Event Type          | Parade               
 Event Bor

In [98]:
spark.stop()